# 01 — Why fine-tune? From the mini-GPT to Mistral 7B

**Chapter 3 · Fine-tuning a large model** · suggested time: 20 min · CPU only · [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zenocode-org/ai-masterclass/blob/main/en/chapter_3/01_why_finetune.ipynb)

![Chapter 3 roadmap](https://raw.githubusercontent.com/zenocode-org/ai-masterclass/main/en/chapter_3/figures/00_roadmap.png)

In chapter 2, we **trained from scratch** a 200,000-parameter mini-GPT on a single corpus. It learned spelling, layout, a few words. But it does not "know" anything about the world, and on Kaamelott (4,000 characters) it mostly recited.

Real language models are first trained on **trillions of tokens** (pre-training), which costs millions of euros. Nobody redoes this step to adapt a model to their own use: we start from a pre-trained model and **fine-tune** it, that is, we **continue training** on our data.

**The common thread of this chapter:** how do you turn a 7-billion-parameter general-purpose model into a model that imitates the dialogues of *Kaamelott*, Plato or Shakespeare, **without destroying it**, with a single GPU?

**By the end of this notebook you will be able to:**
1. place Mistral 7B relative to the mini-GPT of chapter 2: **same architecture**, different scale;
2. explain what a **BPE** tokenizer changes compared with the character tokenizer;
3. choose between *prompting*, RAG and fine-tuning;
4. write the **supervised fine-tuning** (SFT) loss: the cross-entropy of chapter 2, **masked** to the response;
5. tell apart three objectives: domain adaptation, conversational style, instruction following.

In [1]:
# Setup. On Colab: we fetch the course repository (data + shared modules).
import os, sys
if "google.colab" in sys.modules and not os.path.exists("finetune.py"):
    !git clone -q --depth 1 https://github.com/zenocode-org/ai-masterclass
    %cd ai-masterclass/en/chapter_3
sys.path.insert(0, "../chapter_2")   # minigpt.py from chapter 2

import math, random, warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # harmless tqdm warning
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

import finetune as ft   # finetune.py, next to this notebook
torch.manual_seed(1337)
print("PyTorch", torch.__version__)
import minigpt   # the mini-GPT from chapter 2
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"   # download: only the config + tokenizer (a few MB)

PyTorch 2.15.0+cu132


## 1. Same architecture, different scale

Mistral 7B is a **decoder-only** model, exactly like our `GPTLanguageModel` from notebook 08 of chapter 2. To check this without downloading 14 GB of weights, we build the model on the **`meta`** device: PyTorch creates the structure and the tensor shapes, but **allocates no memory** for the values.

In [2]:
config = AutoConfig.from_pretrained(MODEL_ID)
with torch.device("meta"):                      # structure only, 0 bytes of weights
    mistral = AutoModelForCausalLM.from_config(config)

n_mistral = sum(p.numel() for p in mistral.parameters())
mini = minigpt.GPTLanguageModel(vocab_size=65, block_size=64, n_embd=64, n_head=4, n_layer=4)
n_mini = sum(p.numel() for p in mini.parameters())
print(f"mini-GPT (chapter 2): {n_mini:>15,} parameters")
print(f"Mistral 7B          : {n_mistral:>15,} parameters  (x {n_mistral / n_mini:,.0f})")
print()
print(mistral)

mini-GPT (chapter 2):         211,777 parameters
Mistral 7B          :   7,248,023,552 parameters  (x 34,225)

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32768, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        

### Reading the printout with chapter 2 eyes

| Chapter 2 (`minigpt.py`) | Mistral 7B | What changes |
|---|---|---|
| `token_embedding_table` (65 × 64) | `embed_tokens` (32,768 × 4,096) | vocabulary of **subwords** (BPE, §2) |
| `position_embedding_table` | — | positions encoded by **rotating** q and k (RoPE), inside attention |
| `Head.query / key / value` | `q_proj / k_proj / v_proj` | all heads in **a single** matrix; 32 query heads but only 8 shared key/value heads (*grouped-query attention*), hence `k_proj` is 4,096 → 1,024 |
| `MultiHeadAttention.proj` | `o_proj` | identical |
| `FeedForward`: C → 4C → ReLU → C | `gate_proj`, `up_proj`, `down_proj` | **SwiGLU** variant: 4,096 → 14,336 → 4,096 |
| `LayerNorm` | `RMSNorm` | normalization without re-centering |
| `Block` × 4 | `MistralDecoderLayer` × 32 | 8 times deeper |
| `lm_head` | `lm_head` | identical |
| `block_size = 64` characters | 32,768 tokens | context ~500 times longer |

**The problem stays the same**: predict the next token, with a cross-entropy. Everything you built in chapter 2 is in this model; only the scale and a few modernized building blocks change.

Where are the 7 billion? Let's count by hand with the dimensions from `config.json`.

In [3]:
print({k: config.to_dict()[k] for k in ft.MISTRAL_7B})
assert all(config.to_dict()[k] == v for k, v in ft.MISTRAL_7B.items())

counts = ft.mistral_parameters(ft.MISTRAL_7B)
for name, n in counts.items():
    print(f"{name:20s} {n:>15,}  ({n / n_mistral:6.1%})")
print(f"{'total':20s} {sum(counts.values()):>15,}   <- same as the PyTorch count: {sum(counts.values()) == n_mistral}")
print()
for name, (d_in, d_out) in ft.linear_shapes().items():
    print(f"  {name:10s} {d_in:>6} -> {d_out:<6} : {d_in * d_out:>11,} parameters per layer")

{'vocab_size': 32768, 'hidden_size': 4096, 'intermediate_size': 14336, 'num_hidden_layers': 32, 'num_attention_heads': 32, 'num_key_value_heads': 8}
embedding                134,217,728  (  1.9%)
blocks (linear)        6,979,321,856  ( 96.3%)
norms                        266,240  (  0.0%)
lm_head                  134,217,728  (  1.9%)
total                  7,248,023,552   <- same as the PyTorch count: True

  q_proj       4096 -> 4096   :  16,777,216 parameters per layer
  k_proj       4096 -> 1024   :   4,194,304 parameters per layer
  v_proj       4096 -> 1024   :   4,194,304 parameters per layer
  o_proj       4096 -> 4096   :  16,777,216 parameters per layer
  gate_proj    4096 -> 14336  :  58,720,256 parameters per layer
  up_proj      4096 -> 14336  :  58,720,256 parameters per layer
  down_proj   14336 -> 4096   :  58,720,256 parameters per layer


**96% of the parameters** are in the linear layers of the 32 blocks, and two thirds in the FeedForwards. That is where LoRA (notebook 03) and 4-bit quantization (notebook 04) will act.

---
## 2. The tokenizer: subwords instead of characters

In chapter 2, one token = one character (vocabulary of 65 to ~100 symbols). Mistral uses a **BPE** (*byte-pair encoding*) tokenizer with 32,768 tokens: frequent character sequences become a single token.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("vocabulary size:", len(tokenizer))

sentences = {
    "kaamelott": "PERCEVAL: Merde j’ai plus d’pierre ! Qu’est c’qu’on fait ?",
    "plato": "SOCRATE: Eh bien donc, dis à nos juges qui est-ce qui est capable de rendre les jeunes gens meilleurs.",
    "shakespeare": "First Citizen: Before we proceed any further, hear me speak.",
}
for name, s in sentences.items():
    toks = tokenizer.tokenize(s)
    print(f"\n{name}: {len(s)} characters -> {len(toks)} tokens")
    print(toks)

vocabulary size: 32768

kaamelott: 58 characters -> 25 tokens
['▁PER', 'CE', 'VAL', ':', '▁Mer', 'de', '▁j', '’', 'ai', '▁plus', '▁d', '’', 'p', 'ierre', '▁!', '▁Qu', '’', 'est', '▁c', '’', 'qu', '’', 'on', '▁fait', '▁?']

plato: 102 characters -> 32 tokens
['▁SO', 'CR', 'ATE', ':', '▁E', 'h', '▁bien', '▁donc', ',', '▁dis', '▁à', '▁nos', '▁jug', 'es', '▁qui', '▁est', '-', 'ce', '▁qui', '▁est', '▁capable', '▁de', '▁rend', 're', '▁les', '▁je', 'unes', '▁g', 'ens', '▁meille', 'urs', '.']

shakespeare: 60 characters -> 14 tokens
['▁First', '▁Cit', 'izen', ':', '▁Before', '▁we', '▁proceed', '▁any', '▁further', ',', '▁hear', '▁me', '▁speak', '.']


- `▁` marks a space before the word: spaces are part of the tokens.
- Frequent words (`▁Before`, `▁dialogue`, `▁capable`) are **a single token**; less frequent words (`▁jug|es`, *judges*), slangy elisions (`▁d|’|p|ierre`, *stone*) and capitalized names (`▁SO|CR|ATE`) are cut into pieces.
- English is a bit more "compact" than French: the tokenizer saw more English while it was built.

Let's measure it on an excerpt of each corpus: how many characters per token, on average?

In [5]:
print(f"{'corpus':12s} {'characters':>10s} {'BPE tokens':>10s} {'char/token':>10s}")
for name in ["kaamelott", "plato", "shakespeare"]:
    excerpt = ft.load_text(name)[:20_000]
    n_tok = len(tokenizer(excerpt).input_ids)
    print(f"{name:12s} {len(excerpt):10,} {n_tok:10,} {len(excerpt) / n_tok:10.2f}")

corpus       characters BPE tokens char/token
kaamelott         4,237      1,545       2.74
plato            20,000      6,902       2.90
shakespeare      20,000      6,430       3.11


One token ≈ 3 characters (fewer for Kaamelott's slang). Practical consequence: a context of 512 tokens ≈ 1,500 characters, that is **more than 20 times** the `block_size` of our mini-GPT. And each token already carries meaning: the embedding of `▁capable` was learned on billions of sentences.

> **The apostrophe trap, BPE version.** In chapter 2, the Kaamelott tokenizer did not know `'`. Here, `'` and `’` both exist: nothing is impossible to write, but they are **not the same tokens**. The model will have to learn that Kaamelott uses `’`.

---
## 3. Three ways to adapt a model

| Method | What we change | When to use it | Limitation |
|---|---|---|---|
| **Prompting** (instruction, examples in the prompt) | nothing, just the input | quick tests, few examples | the style fades quickly, limited context, cost on every call |
| **RAG** (we paste relevant documents into the prompt) | nothing, we add a document search | up-to-date **knowledge**, citing sources | does not learn a style or a format |
| **Fine-tuning** | the **weights** | learning a **style**, a format, a behavior, a task | data to prepare, GPU, risk of **forgetting** |

Our goal — speaking like Perceval, like Socrates in Cousin's translation or like a Shakespeare character — is a **style**: a textbook case for fine-tuning. To make a model answer questions about your university's regulations, RAG would be the right tool.

---
## 4. The supervised fine-tuning (SFT) loss

![Loss mask](https://raw.githubusercontent.com/zenocode-org/ai-masterclass/main/en/chapter_3/figures/01_loss_mask.png)

An SFT example is a pair **(prompt x, response y)**. We concatenate the two, pass the sequence through the model, and compute the **same cross-entropy as in chapter 2**… but **only on the response tokens**:

$$\mathcal{L}(\theta) = -\sum_{t=1}^{T} \log p_\theta(y_t \mid y_{<t}, x)$$

Why not count the prompt? Because we do not want to teach the model to **write instructions** or to copy the context: we want it to learn **to respond**.

In PyTorch, the mask is a simple **label `-100`**: `F.cross_entropy` ignores these positions (`ignore_index=-100` by default).

In [6]:
torch.manual_seed(0)
V, T = 10, 8
logits = torch.randn(1, T, V)                                    # model output (B=1, T=8, V=10)
labels = torch.tensor([[-100, -100, -100, -100, 3, 7, 1, 2]])   # 4 masked prompt tokens, 4 response tokens

masked_loss = F.cross_entropy(logits.view(-1, V), labels.view(-1))       # ignore_index=-100 by default
by_hand = F.cross_entropy(logits[0, 4:], labels[0, 4:])                  # only the last 4 positions
print(f"loss with -100 mask      : {masked_loss:.4f}")
print(f"loss on the response only: {by_hand:.4f}")
assert torch.allclose(masked_loss, by_hand)

loss with -100 mask      : 3.6602
loss on the response only: 3.6602


### On a real Kaamelott example

"Instruct" models are trained with a **conversation format** (*chat template*). For Mistral: `<s>[INST] user message [/INST] response</s>`. Let's build an example and see which tokens count in the loss.

In [7]:
prompt = [{"role": "user", "content": "Continue ce dialogue de Kaamelott avec la réplique suivante.\n\n"
                                      "BOHORT: La femelle lièvre, c’est la hase."}]
response = [{"role": "assistant", "content": "KARADOC: Ah non, moi je connais que le cri."}]

prompt_ids = tokenizer.apply_chat_template(prompt, add_generation_prompt=True)["input_ids"]
full_ids = tokenizer.apply_chat_template(prompt + response)["input_ids"]
assert full_ids[:len(prompt_ids)] == prompt_ids      # the prompt is a PREFIX of the full example

labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
print(f"{len(full_ids)} tokens, of which {len(full_ids) - len(prompt_ids)} count in the loss\n")
for i, l in zip(full_ids, labels):
    mark = "   " if l == -100 else " ✔ "
    print(f"{mark}{tokenizer.convert_ids_to_tokens(i)!r}", end="  " if l == -100 else "\n")

59 tokens, of which 19 count in the loss

   '<s>'     '[INST]'     '▁Contin'     'ue'     '▁ce'     '▁dialogue'     '▁de'     '▁Ka'     'am'     'el'     'ott'     '▁avec'     '▁la'     '▁ré'     'pl'     'ique'     '▁suiv'     'ante'     '.'     '<0x0A>'     '<0x0A>'     'BO'     'H'     'ORT'     ':'     '▁La'     '▁fem'     'elle'     '▁li'     'è'     'vre'     ','     '▁c'     '’'     'est'     '▁la'     '▁h'     'ase'     '.'     '[/INST]'   ✔ '▁K'
 ✔ 'AR'
 ✔ 'AD'
 ✔ 'OC'
 ✔ ':'
 ✔ '▁Ah'
 ✔ '▁non'
 ✔ ','
 ✔ '▁mo'
 ✔ 'i'
 ✔ '▁je'
 ✔ '▁con'
 ✔ 'na'
 ✔ 'is'
 ✔ '▁que'
 ✔ '▁le'
 ✔ '▁cri'
 ✔ '.'
 ✔ '</s>'


The ✔ tokens are the response, **including `</s>`**: the model also learns **when to stop**. Libraries do this masking for us (TRL: `completion_only_loss=True`, notebook 05), but it is exactly this computation.

*(In the example, Bohort says "The female hare is called the* hase*", and Karadoc replies "Ah no, all I know is the noise it makes.")*

> **Off by one.** In chapter 2, `y` was `x` shifted by one character. With Hugging Face, we pass `labels` **aligned** with `input_ids`, and the model does the shift internally: position `t` predicts label `t+1`.

---
## 5. One corpus, three objectives

The **same** corpus can serve three different tasks. It is not the architecture that changes, it is **how the (input, target) pairs are built**:

In [8]:
scenes = ft.read_dialogues("kaamelott")
scene = scenes[0]

print("=== 1. Domain adaptation: raw text, loss on everything ===")
print("\n".join(ft.format_line(l) for l in scene[:4]))

print("\n=== 2. Conversational style: context -> next line ===")
ex = ft.build_examples([scene], "kaamelott", k=3, mode="continue")[2]
print("PROMPT  :", ex["prompt"][0]["content"])
print("RESPONSE:", ex["completion"][0]["content"])

print("\n=== 3. Instruction: \"answer as this character\" ===")
ex = ft.build_examples([scene], "kaamelott", k=3, mode="speaker")[2]
print("PROMPT  :", ex["prompt"][0]["content"])
print("RESPONSE:", ex["completion"][0]["content"])

=== 1. Domain adaptation: raw text, loss on everything ===
KARADOC: Bon, je vais essayer de trouver un petit lièvre pour ce soir parce qu’il commence à faire faim !
BOHORT: Je me demande comment vous faîtes, moi je serais incapable de trouver de quoi manger dans la forêt.
KARADOC: Pour attraper des bêtes, il faut imiter les femelles. Là, pour le coup, hop ! La femelle lièvre.
BOHORT: La hase.

=== 2. Conversational style: context -> next line ===
PROMPT  : Continue ce dialogue de Kaamelott avec la réplique suivante.

KARADOC: Bon, je vais essayer de trouver un petit lièvre pour ce soir parce qu’il commence à faire faim !
BOHORT: Je me demande comment vous faîtes, moi je serais incapable de trouver de quoi manger dans la forêt.
KARADOC: Pour attraper des bêtes, il faut imiter les femelles. Là, pour le coup, hop ! La femelle lièvre.
RESPONSE: BOHORT: La hase.

=== 3. Instruction: "answer as this character" ===
PROMPT  : Réponds comme BOHORT dans Kaamelott.

KARADOC: Bon, je vais essayer 

| Objective | What the model learns | Use |
|---|---|---|
| 1. Domain adaptation | the distribution of the text (vocabulary, turns of phrase) | continuing text; this is what chapter 2 did |
| 2. Conversational style | responding to a context **and** choosing who speaks | generating the rest of a scene |
| 3. Instruction | obeying an explicit instruction ("answer as Perceval") | "in character" chatbot |

In this chapter, we use **objective 2** (the file exported in notebook 02). The assignment suggests comparing with objective 3.

---
## 6. What can go wrong

- **Catastrophic forgetting**: by pushing the weights towards Kaamelott, we can damage what the model knew how to do (formal French, answering a question). We will **measure** it in notebook 03.
- **Memorization**: with little data (Kaamelott!), the model can recite the corpus instead of generalizing from it, exactly as in chapter 2.
- **Loss of alignment**: an Instruct model has learned to refuse certain requests. A style fine-tuning can erode this behavior: the evaluation (notebook 06) must include **what we did not want to change**.
- **Memory**: 7 billion parameters do not easily fit on a GPU. Notebooks 03 and 04: LoRA and 4-bit quantization.

## Summary

- Mistral 7B **is** a decoder-only GPT: embeddings, attention + FeedForward blocks, residuals, normalization, `lm_head`. Only the scale and a few building blocks (RoPE, GQA, SwiGLU, RMSNorm) change.
- The BPE tokenizer splits text into **subwords**: ≈ 3 characters per token, vocabulary of 32,768.
- Fine-tuning = **continuing training** with the same cross-entropy, **masked** (`-100`) on the prompt.
- It is **how the examples are built** that defines the learned task.

**Check your understanding**
1. Why is `k_proj` 4 times smaller than `q_proj` in Mistral?
2. What would happen if we did not mask the prompt in the loss?
3. For an assistant that answers questions about the library's opening hours, would you choose RAG or fine-tuning? Why?
4. Why must the `</s>` token be part of the response in the loss?